## Q&A System App in LangGraph

This Q&A System App is built for Q&A over a SQL database. The application gives an LLM access to tools for querying and interacting with the data. It follows the recommendation from LangGraph to load the CSV file into a SQL database instead of directly working with a CSV file. Using SQL requires executing model-generated SQL queries. If instead the graph was directly handling the data from CSV, it would require using a library like Pandas and letting the model execute Python code. Since it is easier to tightly scope SQL connection permissions and sanitize SQL queries than it is to sandbox Python environments, [LangGraph's documentation](https://python.langchain.com/docs/tutorials/sql_qa/) recommends interacting with CSV data via SQL. This is the approach followed in this notebook and analyzed a dataset about products on Amazon.

Using SQL to interact with CSV data is the recommended approach because it is easier to limit permissions and sanitize queries than with arbitrary Python. I already have loaded the CSV file in a SQL database. After that it is possible to use all of the chain and agent-creating techniques outlined in the SQL tutorial provided by LangGraph. In the below I use SQLite.

In [462]:
# Setup
%pip install -qU langgraph langchain langchain-community langchain_groq

Note: you may need to restart the kernel to use updated packages.


In [463]:
# Import libraries
import os
import yaml
from dotenv import load_dotenv
from pathlib import Path

import logging
import csv
import re

import time

import pandas as pd

import getpass

from typing import TypedDict, Dict, Any, List, Optional
from typing_extensions import Annotated

from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langchain_groq import ChatGroq
from langgraph.graph import START, StateGraph

from langchain import hub

from data.questions import queries # Import the dictionary of queries containing the questions and expected results

from utils.metrics_collector import MetricsCollector

from utils.logging import *

In [464]:
# Set the framework that will be benchmarked
FRAMEWORK = "LangGraph"

In [465]:
# Load environment variables
# Returns "True" if successful
load_dotenv()

True

In [466]:
# Load config.yaml file
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [467]:
# Load the LLM
def get_llm(config):
    groq_key = os.getenv("GROQ_API_KEY")
    if not groq_key:
        raise ValueError("GROQ_API_KEY is not set.")

    return ChatGroq(
        model=config['model']['name'],
        api_key=groq_key,
        temperature=config['model']['temperature'],
        # max_tokens=config['model']['max_tokens']
    )

llm = get_llm(config)

In [468]:
# Get API key for GROQ
if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

In [469]:
# Connect to database
db = SQLDatabase.from_uri("sqlite:///data/amazon_cleaned.db")

In [470]:
# Function for extracting the questions from the queries dictionary
def extract_questions(queries):
    """Loops over the queries in the dictionary and returns each question formatted as 'question_X: question'."""
    questions = {}
    
    # Loop over each entry in the queries dictionary
    for key, value in queries.items():
        # Extract the question and store it in the questions dictionary
        questions[key] = value['question']
    
    return questions

# Extract the questions from the queries dictionary
questions = extract_questions(queries)

In [471]:
# Define the LangGraph state of the application
class State(TypedDict):
    question: str
    query: str
    result: str
    answer: str
    numeric: float

In [472]:
# Pull a prompt from the Prompt Hub to instruct the model.
query_prompt_template = hub.pull("langchain-ai/sql-query-system-prompt")

assert len(query_prompt_template.messages) == 1
query_prompt_template.messages[0].pretty_print()

================================ System Message ================================

Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of examples they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Only use the following tables:
{table_info}

Question: {input}


The prompt includes several parameters we will need to populate, such as the SQL dialect and table schemas. LangChain's SQLDatabase object includes methods to help with this. Our write_query step will just populate these parameters and prompt a model to generate the SQL query:

In [473]:
class QueryOutput(TypedDict):
    """Generated SQL query."""

    query: Annotated[str, ..., "Syntactically valid SQL query."]


def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 1,
            "table_info": db.get_table_info(),
            "input": state["question"],
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

In [474]:
# Create a function for executing a SQL Query
def execute_query(state: State):
    """Execute SQL query."""
    execute_query_tool = QuerySQLDatabaseTool(db=db)
    return {"result": execute_query_tool.invoke(state["query"])}

In [475]:
# Generate an answer to the question given the information pulled from the database
def generate_answer(state: State):
    """Answer question using retrieved information as context."""
    prompt = (
        "Given the following user question, corresponding SQL query, "
        "and SQL result, answer the user question.\n\n"
        "\n\n"
        f'Question: {state["question"]}\n'
        f'SQL Query: {state["query"]}\n'
        f'SQL Result: {state["result"]}'
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}

In [476]:
# Extract the numerical value from the answer
# Return "Error" if no numerical value is found
def extract_numerical_value(state: State):
    """Extracts the first numerical value from a string (including decimals)."""
    match = re.search(r'\d+(\.\d+)?', state['answer'])  # Regex to find integers or decimal numbers
    if match:
        return {'numeric': float(match.group())}  # Convert the matched value to a float
    return 'Error retrieving numeric answer' # Return None if no numerical value is found

In [477]:
# Create initial state for the workflow
def create_initial_state() -> State:
    """Create initial state for the workflow.
    
    Returns:
        State: Initial workflow state
    """
    return State(
        question='',
        query='',
        result='None',
        answer='',
        numeric=0.0
    )

In [478]:
# Function to save the result in a CSV file for the question and final numeric answer
def save_results_to_csv(results, csv_filename, iteration):
    n_iteration = iteration + 1

    # Open the CSV file in append mode ('a')
    name = csv_filename + str(n_iteration) + ".csv"
    with open(name, mode='a', newline='') as file:
        writer = csv.writer(file)

        # Write the header if the file is empty (only the first time)
        if file.tell() == 0:
            writer.writerow(["question", "answer"])
            
        n_results = len(results)

        # Write each entry of the results as a new row
        for i in range(n_results):
            writer.writerow(results[i])
        
    print(f"Results for iteration {n_iteration} saved to {name}")

In [ ]:
# Set up logging
logger = setup_logging(framework_name=FRAMEWORK)

In [480]:
# Function to compare the expected results with the generated results
def compare_answers(expected_file, csv_filename):
    # Get number of iterations from config
    num_iterations = config['benchmarks']['iterations']

    total_match_percentages = 0

    # Read the CSV file with expected results into pandas DataFrames
    expected_df = pd.read_csv(expected_file)

    # Read the CSV file with generated results into pandas DataFrames
    for iteration in range(num_iterations):
        name = csv_filename + str(iteration + 1) + ".csv"
        results_df = pd.read_csv(name)
        
        # Ensure the DataFrames have the same number of rows
        if len(expected_df) != len(results_df):
            raise ValueError("The number of rows in the two files do not match.")
        
        # Compare the 'answer' columns of both DataFrames
        matches = (expected_df['answer'] == results_df['answer'])
        
        # Calculate the percentage of matching answers
        match_percentage = (matches.sum() / len(matches)) * 100

        # Add the match percentage to the total match percentages
        total_match_percentages += match_percentage
            
    return total_match_percentages / num_iterations

In [481]:
# Build the graph
def build_graph(
    config: Dict[str, Any],
    logger: logging.Logger,
    llm: Optional[ChatGroq] = None
) -> StateGraph:
    
    """Build the complete agent workflow graph.
    Args:
        config: Configuration dictionary
        logger: Logger instance
        llm: Optional ChatGroq instance (will be created if not provided)
        
    Returns:
        StateGraph: Configured workflow graph
    """
    
    # Create LLM if not provided
    if llm is None:
        raise ValueError("LLM is not provided")

    # Create initial state
    initial_state = create_initial_state()

    # Save the results of each query in a list that will later be saved in a CSV file
    results = []

    # Orchestrate with LangGraph
    graph_builder = StateGraph(State).add_sequence(
        [write_query, execute_query, generate_answer, extract_numerical_value]
        )
    graph_builder.add_edge(START, "write_query")
    graph = graph_builder.compile()

    results = []

    # Loop over the queries in the dictionary with questions
    for key, value in questions.items():
        print(f"Processing question: {key, value}")

        # Execute the graph for the current question        
        for step in graph.stream(
            {"question": value}, stream_mode="updates"
        ):
            print(f"Step: {step}")

        # Append the results to the list
        results.append(
                    (
                        value,  # The question value
                        step['extract_numerical_value']['numeric']     # Safe access to numeric value
                    )
                )
        
    # After looping, print the results to verify
    logger.debug("Results generated")

    # Create final state
    final_state = StateGraph(State)

    return results, initial_state, final_state

In [ ]:
# Run the benchmark process with iterations
def run_benchmark(config: Dict[str, Any], llm: ChatGroq, logger: logging.Logger) -> List[Dict[str, Any]]:
    """Run the benchmark process with iterations"""
    logger.debug("Starting benchmark run...")

    # Get number of iterations from config
    num_iterations = config['benchmarks']['iterations']
    logger.info(f"Running benchmark for {num_iterations} iterations")
    
    # Initialize metrics collector
    metrics = MetricsCollector('/results')

    # Run iterations
    for iteration in range(num_iterations):
        logger.info(f"Starting iteration {iteration + 1}/{num_iterations}")

        metrics.start_iteration(FRAMEWORK, iteration)
        start_time = time.time()

        results, initial_state, final_state = build_graph(config, logger, llm)
        api_latency = time.time() - start_time

        metrics.increment_api_calls()
        metrics.add_api_latency(api_latency)
        
        save_results_to_csv(results, "results", iteration)

        logger.info(f"Completed iteration {iteration + 1}")
        print("Start time", start_time)
        print("Latency time", api_latency)

        metrics.end_iteration()
        metrics.save_iteration(FRAMEWORK)
    
    metrics.save_metrics()
    metrics.generate_plots()

    # Generate metric for calculating the percentage of correct answers
    expected_file = 'data/expected_results.csv'
    file_name = 'results'
    correct_answers = compare_answers(expected_file, file_name)
    print(f"The percentage of correct answers for the framework {FRAMEWORK} is: {correct_answers:.1f}%")

    logger.info("Benchmark completed successfully")

In [483]:
# Main execution function
def main():
    """Main entry point for the application."""
    try:        
        # Run workflow
        run_benchmark(config, llm, logger)  
        
    except Exception as e:
        logger.error(f"Error in workflow: {str(e)}")
        raise

if __name__ == "__main__":
    main()

2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24 16:16:18,619 - LangGraph_benchmark - INFO - Running benchmark for 3 iterations
2025-03-24

Starting iteration 1/3
Processing question: ('question_2', 'How many unique product IDs are there in the table?')
Step: {'write_query': {'query': 'SELECT COUNT(DISTINCT product_id) FROM amazon_cleaned'}}
Step: {'execute_query': {'result': '[(1351,)]'}}
Step: {'generate_answer': {'answer': 'There are 1351 unique product IDs in the table.'}}
Step: {'extract_numerical_value': {'numeric': 1351.0}}
Processing question: ('question_3', 'What is the rating of the first product in the table?')
Step: {'write_query': {'query': 'SELECT rating FROM amazon_cleaned LIMIT 1'}}
Step: {'execute_query': {'result': '[(3.8,)]'}}
Step: {'generate_answer': {'answer': 'The rating of the first product in the table is 3.8.'}}
Step: {'extract_numerical_value': {'numeric': 3.8}}
Processing question: ('question_4', "How many unique values for 'category' are in the database?")
Step: {'write_query': {'query': 'SELECT DISTINCT category FROM amazon_cleaned'}}
Step: {'execute_query': {'result': "[('Car&Motorbike',), ('

c:\Users\tsvet\Desktop\2. Volunteering\(1) Omdena\24.12. AI Agents Inference Benchmarking Challenge\Scenario 1 Amazon Dataset\1. Final\utils\metrics_collector.py:139: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[row][col].set_xticklabels(frameworks, rotation=45)
c:\Users\tsvet\Desktop\2. Volunteering\(1) Omdena\24.12. AI Agents Inference Benchmarking Challenge\Scenario 1 Amazon Dataset\1. Final\utils\metrics_collector.py:139: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[row][col].set_xticklabels(frameworks, rotation=45)
c:\Users\tsvet\Desktop\2. Volunteering\(1) Omdena\24.12. AI Agents Inference Benchmarking Challenge\Scenario 1 Amazon Dataset\1. Final\utils\metrics_collector.py:139: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.

The percentage of correct answers for the framework LangGraph is: 100.0%


In [484]:
import os
from pathlib import Path

def check_directory_permissions(directory: str):
    # Convert the directory path to a Path object
    directory_path = Path(directory)

    # Check if the directory exists
    if not directory_path.exists():
        print(f"Error: The directory {directory} does not exist!")
        return False
    
    # Check if it's a directory
    if not directory_path.is_dir():
        print(f"Error: The path {directory} is not a directory!")
        return False

    # Check if the directory is writable
    if not os.access(directory_path, os.W_OK):
        print(f"Error: The directory {directory} is not writable!")
        return False

    # If all checks pass, return True
    print(f"Directory {directory} exists and is writable.")
    return True

# Call the function to check the '/results' directory
check_directory_permissions('/results')


Directory /results exists and is writable.


True